### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="electric_motor_temperature_prediction",
    dataset_year="2021",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature",
    download_description="""
kaggle datasets download wkirgsn/electric-motor-temperature --unzip \
&& mkdir -p local-data-warehouse/electric_motor_temperature_prediction \
&& mv measures_v2.csv local-data-warehouse/electric_motor_temperature_prediction/
""",
    # References
    academic_reference_bibtex="""@article{kirchgassner2020estimating,
  title={Estimating electric motor temperatures with deep residual machine learning},
  author={Kirchg{\"a}ssner, Wilhelm and Wallscheid, Oliver and B{\"o}cker, Joachim},
  journal={IEEE Transactions on Power Electronics},
  volume={36},
  number={7},
  pages={7480--7488},
  year={2020},
  publisher={IEEE}
}
""",
    academic_reference_bibtex_key="kirchgassner2020estimating",
    license="CC BY-SA 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We select the task to predict permanent magnet temperature from the input features. We create grouped splits on the profile_id, which corresponds to predicting the temperature for an unseen motor run session, thus simulating how the model would be used in reality.

- The original study also incorporates temporal connection within the session. We re-create a time-index (https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature/discussion/147446).
- We also add the derived inputs used in the original study.
- We also add the EWMA/WEMS features with window sizes of [500, 2000, 4000, 8000]. To avoid the problem of having an incorrect EWMA/WEMS state and since not all recording are warmed up (https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature/discussion/117319), we drop the first 500 samples (first span) of each profile to simulate such a warm-up.
"""
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="permanent_magnet_temperature",
    problem_type="regression",
    objective_metric_name="rmse",
    # For grouped data
    group_on="profile_id",
    group_labels="per_sample",
    group_time_on="profile_time_index"
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "measures_v2.csv")
print("Loaded data shape:", df.shape)

# Add time feature
df["profile_time_index"] = df.groupby("profile_id").cumcount() + 1

df = df.rename(columns={
    "pm": "permanent_magnet_temperature",
})

df = df.drop(columns=[
    # Other targets
    "stator_winding",
    "stator_tooth",
    "stator_yoke",
    # excluded as in the original study
    "torque",
])

# Add derived features  (from Table II)
# Voltage magnitude
df["u_s"] = np.sqrt(df["u_d"]**2 + df["u_q"]**2)
# Current magnitude
df["i_s"] = np.sqrt(df["i_d"]**2 + df["i_q"]**2)
# Apparent power
df["S_el"] = df["u_s"] * df["i_s"]
# Joint interaction terms
df["i_s_omega"] = df["i_s"] * df["motor_speed"]
df["S_el_omega"] = df["S_el"] * df["motor_speed"]


def add_grouped_ewm_features(
    input_df: pd.DataFrame,
    group_col: str,
    time_col: str,
    cols: list[str],
    input_spans: list[int],
) -> pd.DataFrame:
    input_df = input_df.sort_values([group_col, time_col]).copy()

    for col in cols:
        for span in input_spans:
            alpha = 2 / (span + 1)

            # EWMA
            input_df[f"{col}_ewma_{span}"] = (
                input_df.groupby(group_col, sort=False)[col]
                  .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
            )

            # EWMS
            input_df[f"{col}_ewms_{span}"] = (
                input_df.groupby(group_col, sort=False)[col]
                  .transform(lambda s: s.ewm(alpha=alpha, adjust=False).std())
            )

    return input_df

input_cols = [
    # Original features
    "u_d", "u_q",
    "i_d", "i_q",
    "coolant", "motor_speed", "ambient",
    # Derived features
    "u_s", "i_s", "S_el", "i_s_omega", "S_el_omega",
]
spans = [500, 2000, 4000, 8000]
df = add_grouped_ewm_features(
    input_df=df,
    group_col="profile_id",
    time_col="profile_time_index",
    cols=input_cols,
    input_spans=spans,
)

# Drop the first 500 samples of each profile to simulate warm-up and avoid incorrect EWMA/WEMS state
df = (
    df.sort_values(["profile_id", "profile_time_index"])
      .loc[lambda d: d.groupby("profile_id").cumcount() >= 500]
      .reset_index(drop=True)
)
df["profile_id"] = df["profile_id"].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle data

Loaded data shape: (1330816, 13)
['u_q', 'coolant', 'u_d', 'motor_speed', 'i_d', 'i_q', 'permanent_magnet_temperature', 'ambient', 'profile_id', 'profile_time_index', 'u_s', 'i_s', 'S_el', 'i_s_omega', 'S_el_omega']


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,296,316
Columns: 111
Use sampling: True (sample size: 129,632)
Get missing and unique counts per column...


missing/unique per-col:   0%|          | 0/111 [00:00<?, ?it/s]

Get example values per column...


examples per-col:   0%|          | 0/111 [00:00<?, ?it/s]

Get numeric feature statistics...


...

cat stats:   0%|          | 0/1 [00:00<?, ?it/s]

Get target stats...
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['S_el', 'u_d_ewma_500', 'S_el_omega', 'u_q_ewma_4000', 'u_q_ewms_2000', 'u_q_ewma_2000', 'u_q_ewms_500', 'u_q_ewma_500', 'u_d_ewms_8000', 'u_d_ewma_8000']
Rows remaining as candidates after top-10 filter: 0 (of 1,296,316)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...


hash cols:   0%|          | 0/111 [00:00<?, ?it/s]

group check:   0%|          | 0/111 [00:00<?, ?it/s]

Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,u_q,coolant,u_d,motor_speed,i_d,i_q,permanent_magnet_temperature,ambient,profile_id,profile_time_index,u_s,i_s,S_el,i_s_omega,S_el_omega,u_d_ewma_500,u_d_ewms_500,u_d_ewma_2000,u_d_ewms_2000,u_d_ewma_4000,u_d_ewms_4000,u_d_ewma_8000,u_d_ewms_8000,u_q_ewma_500,u_q_ewms_500,u_q_ewma_2000,u_q_ewms_2000,u_q_ewma_4000,u_q_ewms_4000,u_q_ewma_8000,u_q_ewms_8000,i_d_ewma_500,i_d_ewms_500,i_d_ewma_2000,i_d_ewms_2000,i_d_ewma_4000,i_d_ewms_4000,i_d_ewma_8000,i_d_ewms_8000,i_q_ewma_500,i_q_ewms_500,i_q_ewma_2000,i_q_ewms_2000,i_q_ewma_4000,i_q_ewms_4000,i_q_ewma_8000,i_q_ewms_8000,coolant_ewma_500,coolant_ewms_500,coolant_ewma_2000,coolant_ewms_2000,coolant_ewma_4000,coolant_ewms_4000,coolant_ewma_8000,coolant_ewms_8000,motor_speed_ewma_500,motor_speed_ewms_500,motor_speed_ewma_2000,motor_speed_ewms_2000,motor_speed_ewma_4000,motor_speed_ewms_4000,motor_speed_ewma_8000,motor_speed_ewms_8000,ambient_ewma_500,ambient_ewms_500,ambient_ewma_2000,ambient_ewms_2000,ambient_ewma_4000,ambient_ewms_4000,ambient_ewma_8000,ambient_ewms_8000,u_s_ewma_500,u_s_ewms_500,u_s_ewma_2000,u_s_ewms_2000,u_s_ewma_4000,u_s_ewms_4000,u_s_ewma_8000,u_s_ewms_8000,i_s_ewma_500,i_s_ewms_500,i_s_ewma_2000,i_s_ewms_2000,i_s_ewma_4000,i_s_ewms_4000,i_s_ewma_8000,i_s_ewms_8000,S_el_ewma_500,S_el_ewms_500,S_el_ewma_2000,S_el_ewms_2000,S_el_ewma_4000,S_el_ewms_4000,S_el_ewma_8000,S_el_ewms_8000,i_s_omega_ewma_500,i_s_omega_ewms_500,i_s_omega_ewma_2000,i_s_omega_ewms_2000,i_s_omega_ewma_4000,i_s_omega_ewms_4000,i_s_omega_ewma_8000,i_s_omega_ewms_8000,S_el_omega_ewma_500,S_el_omega_ewms_500,S_el_omega_ewma_2000,S_el_omega_ewms_2000,S_el_omega_ewma_4000,S_el_omega_ewms_4000,S_el_omega_ewma_8000,S_el_omega_ewms_8000
0,-2.096244,62.223688,1.664608,0.001003,-2.000487,1.098619,77.498236,26.346033,76,8768,2.676781,2.282304,6.109228,2.289900e-03,6.129562e-03,-7.681700,33.894432,-13.084412,56.318832,-8.343476,57.705183,-4.340015,56.511980,22.211271,46.429748,53.005856,52.531843,51.058756,50.909152,45.515545,50.085248,-23.573697,46.402029,-59.706847,62.583651,-58.947646,62.471776,-53.909337,62.031806,7.635789,28.387802,3.152057,58.250874,-0.009774,63.472660,0.191756,65.279577,62.248308,1.012289,64.898807,6.573771,65.555176,10.269212,60.149717,16.892353,956.818779,1795.513671,2337.807946,2168.929170,2248.092343,2141.273858,1997.821703,2127.554420,26.292732,0.290655,26.265771,0.210988,26.175974,0.238199,26.032000,0.289304,32.129543,53.137874,73.189597,59.618137,71.580259,58.936519,64.913978,59.596189,27.739290,52.945824,74.513296,73.017123,76.123210,74.904216,71.367421,76.774950,3329.223425,6908.195966,8816.089050,9243.507212,8812.747252,9333.239652,8134.737250,9393.879517,107834.945862,227674.028276,287834.474368,311944.039037,282360.115089,311999.144228,255549.256997,309857.469639,1.388807e+07,2.935391e+07,3.641664e+07,4.047250e+07,3.548600e+07,4.050924e+07,3.196538e+07,4.010690e+07
1,-2.173978,74.151566,0.222524,0.003956,-2.002110,1.096618,61.762292,25.092981,63,5902,2.185337,2.282765,4.988610,9.030746e-03,1.973522e-02,3.041417,27.035296,-4.161962,61.560295,-6.861791,69.119375,-6.226701,67.655363,7.796962,27.281483,40.622455,42.781444,50.308691,42.647332,46.185097,44.716979,-10.592248,31.571030,-48.331580,60.780923,-61.580542,63.564765,-56.931277,65.070159,-3.872850,35.139565,0.470308,81.199107,5.301869,90.787242,6.637607,88.415767,59.891418,10.782185,48.812016,10.486852,43.499313,10.861997,38.110591,11.016667,336.482200,938.203975,1567.383648,1613.048426,1970.365755,1652.176466,1812.880386,1749.009722,25.099772,0.096264,25.114710,0.122790,25.062778,0.145831,24.985583,0.162289,15.832692,35.974898,63.920981,56.571265,78.502849,54.780455,71.881252,58.446980,17.389276,45.342868,77.510413,81.326810,96.013733,82.875031,87.390350,86.364499,1592.033664,5361.396601,8411.954174,10209.444868,10730.494083,10539.996273,9841.826750,10817.198686,40346.324874,145094.174096,209923.683232,271092.522343,270117.401672,284016.725248,249596.625905,291665.569730,4.873236e+06,1.872292e+07,2.571203e+07,

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,profile_id,category,0.0,0.0,69.0,"20, 6, 65, 66, 18, 13, 27, 56, 4, 58"
1,u_q,float64,0.0,0.0,1237646.0,"131.8004, 8.1426, 131.7986, 132.0084, 131.7984, 131.7957, 131.8011, 8.1419, 119.5759, 131.7592"
2,coolant,float64,0.0,0.0,1082713.0,"56.1055, 35.7982, 81.4896, 61.222, 81.4896, 61.222, 15.4908, 30.7213, 35.7982, 15.4908"
3,u_d,float64,0.0,0.0,1261982.0,"-129.9406, -7.6112, -128.0483, -11.633, -7.6103, -130.0341, -7.6103, -130.0807, -11.6377, -7.6161"
4,motor_speed,float64,0.0,0.0,754389.0,"4999.9463, 4999.9458, 4499.9561, 4999.9453, 4999.9478, 4999.9468, 4499.957, 4999.9473, 4999.9443, 4999.9448"
5,i_d,float64,0.0,0.0,1019349.0,"-43.5117, -43.5118, -43.5116, -43.5116, -43.5117, -43.5116, -43.5114, -43.5118, -43.5121, -43.5121"
6,i_q,float64,0.0,0.0,996985.0,"132.6176, 132.6176, 132.6176, 132.6174, 132.6177, 132.6177, 132.6179, 132.6176, 132.6176, 132.6176"
7,permanent_magnet_temperature,float64,0.0,0.0,1282892.0,"44.1124, 38.4401, 82.8469, 31.8331, 40.0473, 41.5088, 88.7856, 72.4434, 28.4124, 36.8684"
8,ambient,float64,0.0,0.0,1164004.0,"19.8506, 26.3845, 26.3845, 26.3845, 19.8506, 26.3845, 19.8506, 19.8506, 19.8506, 26.3845"
9,u_s,float64,0.0,0.0,1296314.0,"131.2528, 130.9348, 17.9342, 131.2762, 17.9544, 12.1076, 0.9601, 1.1741, 132.1215, 129.9521"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
u_q,129632.0,5.423378e+01,4.423986e+01,-2.491796e+01,1.322492e+02
coolant,129632.0,3.641024e+01,2.186564e+01,1.199671e+01,9.373608e+01
u_d,129632.0,-2.466753e+01,6.317037e+01,-1.314386e+02,1.313946e+02
motor_speed,129632.0,2.197430e+03,1.856846e+03,-2.426863e+02,6.000015e+03
i_d,129632.0,-6.824552e+01,6.485284e+01,-2.776520e+02,6.458024e-03
i_q,129632.0,3.661469e+01,9.175560e+01,-2.929667e+02,3.017075e+02
permanent_magnet_temperature,129632.0,5.910639e+01,1.871198e+01,2.202800e+01,1.135573e+02
ambient,129632.0,2.460547e+01,1.907550e+00,9.934283e+00,3.012359e+01
profile_time_index,129632.0,1.230782e+04,8.873765e+03,5.010000e+02,4.396600e+04
u_s,129632.0,8.151762e+01,5.340697e+01,3.604401e-06,1.338712e+02


In [7]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column     rank                   
profile_id 1       20   4439  3.42
           2        6   3970  3.06
           3       65   3937  3.04
           4       66   3647  2.81
           5       18   3642  2.81

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.032,-0.589,350.138,0.121,log,1350022.6,2.968767e+09,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=1, n_splits=1, test_size=250000


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using label-per-sample grouped splits.
Repeat 0, Fold 0:
            Train N: 968996, Test N: 327320
            Target Distribution:
            	Train target distribution: 62.405901049380915
            	Test target distribution: 49.361947978382396
            Group Distribution profile_id:
            	Train: 55
            	Test: 14
            


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to electric_motor_temperature_prediction/019d459a-334a-7077-812d-776a08a70ee2
019d459a-334a-7077-812d-776a08a70ee2
df2dd9a82429ad3ab0e89978ae49945abecf750f65c2699e3267042ea7b131b3
